[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/karzit/temp/blob/master/notebooks/03_classification/03_classification_solutions.ipynb)

# 03. Classification — 연습 문제 해설

[03_classification.ipynb](03_classification.ipynb) 끝의 연습 문제 2개에 대한 정답 코드와 해설입니다. **먼저 직접 시도해본 뒤** 참고하세요.

In [ ]:
import sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !pip install -q scikit-learn numpy matplotlib

import numpy as np
import matplotlib.pyplot as plt

## 연습 1. lr을 바꿔가며 Cross-Entropy Cost 수렴 속도 비교

In [ ]:
rng = np.random.default_rng(1)
hours = np.linspace(0, 10, 60)
prob_pass = 1 / (1 + np.exp(-(hours - 5)))
passed = (rng.random(len(hours)) < prob_pass).astype(float)

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def train_logreg(hours, passed, lr, epochs=500):
    W, b = 0.0, 0.0
    m = len(hours)
    history = []
    eps = 1e-7
    for _ in range(epochs):
        h = sigmoid(W * hours + b)
        cost = -np.mean(passed * np.log(h + eps) + (1 - passed) * np.log(1 - h + eps))
        history.append(cost)
        dW = np.mean((h - passed) * hours)
        db = np.mean(h - passed)
        W -= lr * dW
        b -= lr * db
    return W, b, history

results = {}
for lr in [0.01, 0.1, 1.0]:
    W, b, hist = train_logreg(hours, passed, lr)
    results[lr] = hist
    print(f"lr={lr:<5} 최종 cost={hist[-1]:.4f}  W={W:.3f} b={b:.3f}")

In [ ]:
plt.figure(figsize=(7, 4))
for lr, hist in results.items():
    plt.plot(hist, label=f"lr={lr}")
plt.xlabel("epoch")
plt.ylabel("cross-entropy cost")
plt.title("학습률에 따른 Cross-Entropy 수렴 속도")
plt.legend()
plt.show()

**해설**
- `lr=0.01`: 500 epoch로는 부족해서 cost가 상대적으로 높은 채로 남습니다 — 더 학습시키면 계속 내려갑니다.
- `lr=0.1`(원본 기본값): 빠르고 안정적으로 수렴합니다.
- `lr=1.0`: 이 문제 규모에서는 더 빠르게 수렴하지만, 데이터/스케일에 따라 진동(oscillation)이 생길 수 있습니다. 실무에서는 보통 `lr=0.1`처럼 중간값에서 시작해 조금씩 조정합니다.
- Linear Regression과 마찬가지로 Logistic Regression도 Gradient Descent 기반이라 학습률 트레이드오프가 동일하게 적용됩니다.

## 연습 2. StandardScaler를 빼고 학습하면?

In [ ]:
import warnings
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.exceptions import ConvergenceWarning

iris = load_iris()
X_train, X_test, y_train, y_test = train_test_split(
    iris.data, iris.target, test_size=0.2, random_state=0, stratify=iris.target
)

# (A) 스케일링 없이 원본 값 그대로 학습
with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter("always", ConvergenceWarning)
    clf_raw = LogisticRegression(max_iter=200)
    clf_raw.fit(X_train, y_train)
    n_warnings = sum(1 for warning in w if issubclass(warning.category, ConvergenceWarning))
acc_raw = accuracy_score(y_test, clf_raw.predict(X_test))

# (B) StandardScaler 적용
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)
clf_scaled = LogisticRegression(max_iter=200)
clf_scaled.fit(X_train_s, y_train)
acc_scaled = accuracy_score(y_test, clf_scaled.predict(X_test_s))

print(f"스케일링 없음   : accuracy={acc_raw:.4f}  ConvergenceWarning 발생={n_warnings > 0}  (n_iter={clf_raw.n_iter_})")
print(f"StandardScaler : accuracy={acc_scaled:.4f}  (n_iter={clf_scaled.n_iter_})")

**해설**
- Iris 데이터는 모든 특성이 이미 비슷한 단위(cm)라서, 스케일링 없이도 **최종 정확도 자체는 크게 나쁘지 않을 수 있습니다**.
- 하지만 `n_iter_`(실제 사용된 반복 횟수)를 비교해보면, 스케일링 없이는 최적화가 수렴하는 데 더 오래 걸리거나(`max_iter`에 도달해 `ConvergenceWarning`이 뜰 수 있음) 경사가 특성마다 다른 스케일로 왜곡되어 학습이 비효율적입니다.
- 특성 간 단위 차이가 큰 데이터(예: '나이'와 '연봉'을 같이 쓰는 경우)에서는 스케일링 유무가 정확도에도 훨씬 크게 영향을 줍니다. **스케일링은 항상 습관적으로 적용하는 것이 안전**합니다.